# Session 4 — Hubs, Done Properly

**Goal of this session:** go beyond raw degree — the only hub measure the first course had — to a pair of numbers that actually tells you *what kind* of hub a node is.

*Network Neuroscience in Python, session 4 of 10.*

## Why this matters

In the first course, "hub" meant "high degree", full stop. That's a start, but it throws away useful information: a node can have high degree because it's the best-connected member of one tight-knit community, or because it's the one node stitching several different communities together. Those are functionally very different roles, and a single degree number can't tell them apart. This session adds the two numbers that can, and gives you the classic vocabulary — connector hub, provincial hub, peripheral node — for talking about the difference.

## The toy network and its communities

We need last session's community assignment as an input here, so we rebuild both.

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt


def generate_toy_network(n_per_module=5, n_modules=2, p_within=0.7, p_between=0.05,
                          add_connector=True, connector_frac=0.6, seed=0):
    """Same function as session 1. Not a brain."""
    rng = np.random.default_rng(seed)
    G = nx.Graph()
    node_id = 0
    modules = []
    for m in range(n_modules):
        members = []
        for _ in range(n_per_module):
            G.add_node(node_id, module=m)
            members.append(node_id)
            node_id += 1
        modules.append(members)
    for m in range(n_modules):
        members = modules[m]
        for i in range(len(members)):
            for j in range(i + 1, len(members)):
                if rng.random() < p_within:
                    G.add_edge(members[i], members[j])
    for m1 in range(n_modules):
        for m2 in range(m1 + 1, n_modules):
            for u in modules[m1]:
                for v in modules[m2]:
                    if rng.random() < p_between:
                        G.add_edge(u, v)
    if add_connector:
        connector = node_id
        G.add_node(connector, module="connector")
        n_link = max(1, round(connector_frac * n_per_module))
        for members in modules:
            chosen = rng.choice(members, size=min(n_link, len(members)), replace=False)
            for other in chosen:
                G.add_edge(connector, int(other))
    return G


G = generate_toy_network(seed=0)
communities = nx.community.louvain_communities(G, resolution=1.0, seed=0)
print(f"{len(communities)} communities:", [sorted(c) for c in communities])

true_module = nx.get_node_attributes(G, "module")
known_connector = [n for n, m in true_module.items() if m == "connector"][0]
print("the node we built as a connector hub:", known_connector)

## Participation coefficient: does this node spread its connections around?

For node *i* with degree $k_i$, split its neighbours by which community they belong to. If $k_{i,s}$ is how many of node *i*'s neighbours sit in community *s*, the **participation coefficient** is:

$$P_i = 1 - \sum_s \left(\frac{k_{i,s}}{k_i}\right)^2$$

If every one of node *i*'s neighbours is in the same single community, $P_i = 0$ — all its eggs are in one basket. If its neighbours are spread perfectly evenly across many communities, $P_i$ climbs toward 1. It's the same shape as the Gini–Simpson diversity index, applied to "which community is this edge going to".

In [ ]:
def participation_coefficient(G, communities):
    node_to_comm = {n: ci for ci, members in enumerate(communities) for n in members}
    P = {}
    for i in G.nodes():
        k_i = G.degree(i)
        if k_i == 0:
            P[i] = 0.0
            continue
        neighbour_comms = [node_to_comm[j] for j in G.neighbors(i)]
        total = sum((neighbour_comms.count(c) / k_i) ** 2 for c in set(neighbour_comms))
        P[i] = 1 - total
    return P


P = participation_coefficient(G, communities)
for n in sorted(G.nodes(), key=lambda n: -P[n]):
    print(f"node {n:2d}  degree={G.degree(n)}  P={P[n]:.2f}")

## Within-module degree z-score: is this node unusually well-connected *inside* its own community?

Participation coefficient alone can't distinguish "a node with 2 connections, one in each of two communities" from "a node with 20 connections, ten in each" — both spread evenly, but only one of them is a hub by any reasonable definition. So we add a second number: within each community, count how many of a node's edges stay inside that same community (its within-module degree), then z-score that count *against the other members of the same community*.

$$z_i = \frac{k_i^{\text{within}} - \bar{k}^{\text{within}}_{c_i}}{\sigma^{\text{within}}_{c_i}}$$

A node with $z_i > 0$ has more within-community connections than a typical member of its own community; $z_i < 0$ means fewer.

In [ ]:
def within_module_degree_zscore(G, communities):
    node_to_comm = {n: ci for ci, members in enumerate(communities) for n in members}
    z = {}
    for members in communities:
        within_deg = {i: sum(1 for j in G.neighbors(i) if node_to_comm[j] == node_to_comm[i])
                      for i in members}
        vals = np.array(list(within_deg.values()), dtype=float)
        mean, std = vals.mean(), vals.std()
        for i in members:
            z[i] = 0.0 if std == 0 else (within_deg[i] - mean) / std
    return z


Z = within_module_degree_zscore(G, communities)
for n in sorted(G.nodes(), key=lambda n: -Z[n]):
    print(f"node {n:2d}  P={P[n]:.2f}  z={Z[n]:+.2f}")

## Hub cartography (Guimerà & Amaral, 2005)

Plot every node on participation coefficient (x-axis) against within-module z-score (y-axis), and the plane divides into named regions:

- **Peripheral nodes** — low P, low z: connected mostly to their own community, and not even especially well-connected there.
- **Provincial hubs** — low P, high z: the well-connected core of a single community, but not reaching outside it.
- **Connector hubs** — high P, any z: nodes that bridge multiple communities, regardless of exactly how central they are within any one of them.
- **Kinless / global hubs** — very high P, high z: rare in practice, connected everywhere and central everywhere.

This is exactly the vocabulary used in the human connectome literature to talk about which regions integrate information across the brain's systems versus which ones specialise within one.

In [ ]:
def cartography_role(p, z):
    if p < 0.3:
        return "peripheral" if z < 2.0 else "provincial hub"
    else:
        return "connector hub" if z < 2.0 else "kinless hub"


roles = {n: cartography_role(P[n], Z[n]) for n in G.nodes()}
role_colours = {
    "peripheral": "#a0aec0",
    "provincial hub": "#2b6cb0",
    "connector hub": "#38a169",
    "kinless hub": "#c53030",
}

fig, ax = plt.subplots(figsize=(8.5, 6.5))
for role, colour in role_colours.items():
    xs = [P[n] for n in G.nodes() if roles[n] == role]
    ys = [Z[n] for n in G.nodes() if roles[n] == role]
    ax.scatter(xs, ys, s=160, color=colour, edgecolor="white", linewidth=1.2,
               label=role, zorder=3)

ax.axvline(0.3, color="gray", linestyle="--", linewidth=1, zorder=1)
ax.axhline(2.0, color="gray", linestyle="--", linewidth=1, zorder=1)

# annotate the node we built as a connector
ax.scatter([P[known_connector]], [Z[known_connector]], s=420, facecolors="none",
           edgecolors="#dd6b20", linewidths=3, zorder=4)
ax.annotate("the node we built\nas a connector hub", (P[known_connector], Z[known_connector]),
            xytext=(15, 15), textcoords="offset points", fontsize=11, color="#dd6b20",
            fontweight="bold")

for n in G.nodes():
    ax.annotate(str(n), (P[n], Z[n]), xytext=(6, -3), textcoords="offset points", fontsize=9)

ax.set_xlabel("participation coefficient", fontsize=12)
ax.set_ylabel("within-module degree z-score", fontsize=12)
ax.set_title("Hub cartography: what kind of hub is each node?", fontsize=13)
ax.legend(fontsize=10, loc="upper left")
plt.tight_layout()
plt.show()

print(f"\nthe known connector node ({known_connector}): "
      f"P={P[known_connector]:.2f}, z={Z[known_connector]:+.2f}, "
      f"classified as '{roles[known_connector]}'")
print("highest participation coefficient overall:",
      max(G.nodes(), key=lambda n: P[n]), "-> is that the same node?",
      max(G.nodes(), key=lambda n: P[n]) == known_connector)

## Checking the answer

We built one node deliberately to bridge both communities, without giving it an especially high within-community degree. It has the single highest participation coefficient of all eleven nodes, and hub cartography correctly calls it a connector. That agreement is the entire point of testing methods on a network whose ground truth we already know: it means we can trust these formulas on a real connectome, where we won't have that luxury.

Look closely, though, and several other nodes sit close behind it on participation coefficient. With only eleven nodes and five-member modules, there is not much room for a dramatic separation — a real caveat, not just a toy-network quirk. Cartography boundaries drawn from a handful of nodes should be read as "roughly this region", not "precisely this node and no other". The method scales better as networks grow, which is exactly the direction we're heading from session 7 onward.

**Next session:** we zoom out from individual hub roles to a network-wide question — do the high-degree nodes preferentially wire up with each other, more than a null model would predict?